<a href="https://colab.research.google.com/github/Perry-wang-ab741/114-2-ProgramingLanguage/blob/main/%E3%80%8CHW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
!pip install -q google-genai

In [38]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

In [39]:
from google.colab import userdata
from google import genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

In [40]:
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to make predictions or decisions.


In [41]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1V65JtvOG8CsP5rSpCoZ5WjTfMhw1i29N4pv7wW-D3sY/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [42]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [43]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:

        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [44]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：90
已記錄：日期: 2026-04-04, 科目: 國文, 成績: 90

請輸入科目（或輸入 'q' 停止）：英文
請輸入 英文 的成績：85
已記錄：日期: 2026-04-04, 科目: 英文, 成績: 85

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：60
已記錄：日期: 2026-04-04, 科目: 數學, 成績: 60

請輸入科目（或輸入 'q' 停止）：q


In [45]:
new_grades

[['2026-04-04', '國文', 90], ['2026-04-04', '英文', 85], ['2026-04-04', '數學', 60]]

In [46]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，這是一份根據您提供的成績清單所產出的簡單摘要與常見迷思整理，全程不進行任何評分，僅做客觀總結。\n\n---\n\n### 學生成績摘要與常見迷思整理\n\n**一、 簡單摘要**\n\n這份成績清單呈現了學生在2026年4月4日於三個不同科目（國文、英文、數學）的表現。\n\n*   **國文科目：** 獲得 90 分\n*   **英文科目：** 獲得 85 分\n*   **數學科目：** 獲得 60 分\n\n從數據來看，成績分佈介於60分至90分之間。這是一次單一時間點（2026年4月4日）的成績快照，顯示了學生在這些特定科目上的分數表現。\n\n**二、 常見迷思整理**\n\n在看待學生成績時，社會上常存在一些迷思，客觀地理解這些迷思有助於更全面地看待學習成果：\n\n1.  **迷思一：成績是衡量學生能力與潛力的唯一標準。**\n    *   **總結：** 實際情況是，成績只反映了特定時間點在特定評量方式下的表現，學生的綜合能力、學習態度、人際溝通、創造力、解決問題能力等面向，遠非成績數字所能完全涵蓋。\n\n2.  **迷思二：單一科目成績不佳就代表學生該科目學習徹底失敗。**\n    *   **總結：** 單一科目較低的成績應被視為一個具體的學習訊號，可能反映了該科目的某些特定知識點需要加強、學習方法需調整，或評量方式不適合等，而非全盤否定學生在該科目的所有努力或潛力。\n\n3.  **迷思三：成績是完全客觀且絕對公正的。**\n    *   **總結：** 成績會受到多重因素影響，包括考試設計、題型偏好、學生的臨場狀況、甚至老師的評分標準等。客觀性是評量的目標，但實際執行中總會存在變數，且不同評量方式會呈現不同面向的學習成果。\n\n4.  **迷思四：只要分數高就好，學習過程不重要。**\n    *   **總結：** 分數是學習結果的一種呈現，但其背後的學習過程、努力、進步幅度、對知識的理解與應用能力、解決問題的策略等，這些往往比單純的數字更有價值，也是未來持續學習的基礎。\n\n5.  **迷思五：一旦成績定型，學生表現就不會再改變。**\n    *   **總結：** 學習是一個持續動態的過程。透過適當的指導、調整學習策略、投入時間和努力，學生的表現隨時都有進步的空間和可能性。現在的成績不代表未來永遠的表現。\n

In [47]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：90
已記錄：日期: 2026-04-04, 科目: 國文, 成績: 90

請輸入科目（或輸入 'q' 停止）：英文
請輸入 英文 的成績：85
已記錄：日期: 2026-04-04, 科目: 英文, 成績: 85

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：60
已記錄：日期: 2026-04-04, 科目: 數學, 成績: 60

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，這是一份根據您提供的成績資料所做的簡單摘要與常見迷思整理，全程不帶任何評分或判斷：

---

### 學生成績摘要與常見迷思整理

**一、 簡單的成績摘要**

這份成績清單顯示了學生在 **2026年04月04日** 於三個科目的表現：

*   **國文：90分**
*   **英文：85分**
*   **數學：60分**

**總結而言：**
學生在國文和英文科目上取得了較高的分數，均達到85分以上。數學科目則獲得了及格分數（60分）。這份成績呈現了學生在同一評量期間內，不同科目之間的分數差異。

**二、 常見迷思整理**

在檢視學生成績時，人們常會有一些既定的觀念或迷思。以下針對成績的常見迷思進行整理，旨在提供一個更全面的視角，而不對成績本身進行評斷：

1.  **迷思一：成績是衡量學習成果的唯一標準。**
    *   **澄清：** 成績確實反映了學生在特定評估方式（如考試、作業）下的知識掌握程度。然而，學習成果是多面向的，它也包含理解力、應用能力、批判性思維、解決問題能力、創意、團隊合作等，這些不一定能完全透過分數來量化。

2.  **迷思二：高分代表完全理解，低分代表完全不懂。**
    *   **澄清：** 分數有其局限性。高分可能表示學生熟悉考試範圍和作